In [ ]:
import math
import re
from math import exp
from mpmath import erf
from datetime import datetime

# =========================
# 數學工具
# =========================

def N(x):
    return 0.5 * (1 + erf(x / math.sqrt(2)))

def n(x):
    return math.exp(-0.5 * x**2) / math.sqrt(2 * math.pi)

# =========================
# 解析 Option Chain 原始文字
# =========================

def parse_option_chain(text):
    data = {}

    # Strike
    strike_match = re.search(r'\$(\d+\.?\d*)', text)
    if strike_match:
        data["strike"] = float(strike_match.group(1))

    # 到期日
    date_match = re.search(r'(\d+/\d+)', text)
    if date_match:
        expiry_str = date_match.group(1)
        today = datetime.today()
        expiry = datetime.strptime(f"{expiry_str}/{today.year}", "%m/%d/%Y")
        data["days"] = (expiry - today).days

    # Mark price
    mark_match = re.search(r'Mark\s*\$([\d\.]+)', text)
    if mark_match:
        data["mark"] = float(mark_match.group(1))

    # IV
    iv_match = re.search(r'Implied volatility\s*([\d\.]+)%', text)
    if iv_match:
        data["iv"] = float(iv_match.group(1)) / 100

    # Greeks
    greeks = {}
    for greek in ["Delta", "Gamma", "Theta", "Vega", "Rho"]:
        match = re.search(rf'{greek}\s*(-?[\d\.]+)', text)
        if match:
            greeks[greek.lower()] = float(match.group(1))
        else:
            greeks[greek.lower()] = 0.0

    data["greeks"] = greeks

    # Call or Put
    if "Call" in text:
        data["type"] = "call"
    elif "Put" in text:
        data["type"] = "put"

    return data

# =========================
# Parity
# =========================

def calc_put_by_parity(call_price, strike, spot, r, days):
    T = days / 365
    return call_price + strike * exp(-r * T) - spot

def calc_call_by_parity(put_price, strike, spot, r, days):
    T = days / 365
    return put_price - strike * exp(-r * T) + spot

# =========================
# Greeks 預估
# =========================

def estimate_option_from_greeks(base_price, greeks,
                                delta_spot=0,
                                delta_vol=0,
                                delta_rate=0,
                                days_passed=1):

    change = (
        greeks["delta"] * delta_spot +
        0.5 * greeks["gamma"] * delta_spot**2 +
        greeks["theta"] * days_passed +
        greeks["vega"] * delta_vol +
        greeks["rho"] * delta_rate
    )

    return base_price + change

# =========================
# Black Scholes
# =========================

def black_scholes_price(S, K, T, r, sigma, option_type="call"):

    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)

    if option_type == "call":
        return S*N(d1) - K*math.exp(-r*T)*N(d2)
    else:
        return K*math.exp(-r*T)*N(-d2) - S*N(-d1)


In [ ]:
# =========================
# ===== 使用區，需要直接從option chain貼上 =====
# =========================

risk_free_rate = 0.035

spot = 101.50   # 只需要手動輸入現價




# IV = 0要複製


# ===== 直接貼上 Call =====
call_text = """
XYZ 100 Call 12/19
Bid
$2.10 × 5
Mark
$2.20
Ask
$2.30 × 8
Implied volatility
31.50%
The Greeks
Delta
0.5100
Gamma
0.0400
Theta
-0.0300
Vega
0.1200
Rho
0.0500

"""




# ===== 直接貼上 Put =====
put_text = """
XYZ 100 Put 12/19
Bid
$1.80 × 6
Mark
$1.90
Ask
$2.00 × 9
Implied volatility
30.00%
The Greeks
Delta
-0.4900
Gamma
0.0400
Theta
-0.0250
Vega
0.1200
Rho
-0.0450

"""







call_data = parse_option_chain(call_text)
put_data = parse_option_chain(put_text)

days = call_data["days"]
strike = call_data["strike"]

# =========================
# 計算輸出
# =========================

print("===== Parity =====")
print("Call by parity:",
      round(calc_call_by_parity(put_data["mark"], strike, spot, risk_free_rate, days), 4))
print("Put by parity:",
      round(calc_put_by_parity(call_data["mark"], strike, spot, risk_free_rate, days), 4))

print("\n===== Black Scholes =====")
print("Call BS:",
      round(black_scholes_price(spot, strike, days/365,
                                risk_free_rate, call_data["iv"], "call"), 4))

print("Put BS:",
      round(black_scholes_price(spot, strike, days/365,
                                risk_free_rate, put_data["iv"], "put"), 4))


original_spot = strike  # 假設當時接近ATM，可自行改

delta_S = spot - original_spot

print("\n===== Greeks 預估 =====")

print("Call estimate:",
      round(estimate_option_from_greeks(
          call_data["mark"],
          call_data["greeks"],
          delta_spot=delta_S
      ), 4))

print("Put estimate:",
      round(estimate_option_from_greeks(
          put_data["mark"],
          put_data["greeks"],
          delta_spot=delta_S
      ), 4))
